In [1]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import json
import os, os.path

from collections import defaultdict

import torch
import torch.nn as nn
import transformers
import re

from transformers import (AutoTokenizer,
                          AutoModelForCausalLM)
from peft import (PeftModel,
                  PeftConfig)

import spacy
import random
import copy
import nltk
from nltk.tokenize import sent_tokenize
from shutil import copyfile

import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# put your file path here
path_to_nlu_dir = "/content/drive/MyDrive/NLU/Final Project/NLU_FinalProject/"

data_dir = path_to_nlu_dir+"Data/JSONL_Formatted/"
#data_path = "RACE-H/RACE-H_v1_tst.jsonl"
#data_name = 'RACE-H_FineTuneV1_InfoAlterationSent_test' #InfoAlterationSyn_test'
#data_path = "SAT_ACT/SATACT_v3_tst.jsonl"
#data_name = 'SATACT_FineTuneV1_InfoAlterationSent_test'

save_dir = path_to_nlu_dir+"Results/v2_Results/"

model_name = "gpt2-xl" #baseline
#model_name = "Salm00n/gpt2-xl_RACE-H_v1" #v1 race-h fine-tuned
#model_name = "Salm00n/gpt2-xl_SATACT_v1" #v1 sat/act fine-tuned
BATCH_SIZE = 1

In [ ]:
data_list1 = ["RACE-H/RACE-H_v1_trn.jsonl", "RACE-H/RACE-H_v1_dev.jsonl", "RACE-H/RACE-H_v1_tst.jsonl"]
data_list2 = ["SAT_ACT/SATACT_v3_trn.jsonl", "SAT_ACT/SATACT_v3_dev.jsonl", "SAT_ACT/SATACT_v3_tst.jsonl"]

In [4]:
def hltag(data):
    U = set(['just', 'being', 'able', 'over', 'mainly', 'still', 'yet', 'seemed', 'whose', 'based', 'also', 'writer', 'had', 'should', 'to', 'sometimesd', 'has', 'might', 'then', 'very', 'ones', 'whether', 'not', 'during', 'now', 'realize', 'did', 'this', 't', 'each', 'where', 'because', 'doing', 'some', 'likely', 'are', 'further', 'really', 'even', 'what', 'said', 'for', 'lots', 'since', 'please', 'does', 'between', 'probably', 'ever', 'either', 'available', 'be', 'recently', 'however', 'here', 'although', 'by', 'both', 'about', 'anything', 'of', 'could', 'title', 'according', 's', 'or', 'among', 'already', 'suddenly', 'seems', 'simply', 'passage', 'from', 'would', 'whom', 'there', 'been', 'few', 'too', 'was', 'until', 'that', 'but', 'else', 'with', 'than', 'those', 'must', 'showed', 'these', 'will', 'while', 'can', 'were', 'following', 'and', 'do', 'almost', 'is', 'it', 'an', 'as', 'at', 'have', 'seem', 'if', 'again', 'author', 'rather', 'when', 'how', 'other', 'which', 'instead', 'several', 'though', 'may', 'who', 'most', 'such', 'why', 'recent', 'a', 'don', 'especially', 'maybe', 'perhaps', 'so', 'the', 'having', 'nearly'])
    nlp = spacy.load('en_core_web_sm')
    salientPosList = ['NN', 'NNP', 'NNPS', 'NNS', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'JJ', 'JJR', 'JJS', 'RB',
                      'RBR', 'RBS', 'CD', 'FW']  # 21 core pos tag
    output = []
    for i in range(len(data)):
        if not data[i][0] or not isinstance(data[i][0], list):
            print(data[i])
            continue
        article = ' '.join(data[i][0])
        if not article.strip():
            continue
        #print(article)
        article = nlp(article)

        for j in range(len(data[i][1])):
            d = copy.deepcopy(data[i])
            d[0] = []

            question = data[i][1][j]["question"]
            if not question.strip():
                print(data[i])
                continue
            question = nlp(question)

            for k in range(len(data[i][1][j]["choice"])):
                key = set()
                for token in question:
                    if token.tag_ in salientPosList and token.text.lower() not in U:
                        key.add(token.text.lower())
                choice = data[i][1][j]["choice"][k]
                if not isinstance(choice, str) or not choice.strip():
                    print(data[i])
                    continue
                choice = nlp(choice)

                for token in choice:
                    if token.tag_ in salientPosList:
                        key.add(token.text.lower())
                articleatt = []
                for token in article:
                    if token.tag_ in salientPosList and token.text.lower() in key:
                        articleatt += ['[[HL]]']
                        articleatt += [token.text]
                        articleatt += ['[[/HL]]']
                    else:
                        articleatt += [token.text]
            d[0] += [' '.join(articleatt)]
            #print(d[0])
            d[1] = [data[i][1][j]]
            output.append(d)
    return output

In [5]:
def preprocess(data_list):
    splits = ["sft_trn", "sft_dev", "sft_tst"]
    for n in range(len(data_list)):
      print("preprocessing:", data_list[n])
      file_path = data_dir + data_list[n]
      output = []
      d1 = splits[n]

      with open(file_path, "r") as f:
        for line in f:
          data = json.loads(line)
          d = [[data["context"]], [], d1]
          q = {
              "question": data["question"],
              "choice": [data["answerA"], data["answerB"], data["answerC"], data["answerD"]],
              "answer": data[f"answer{data['correct']}"]
          }
          d[1].append(q)
          output.append(d)

        print(d1, "before highlighting:", len(output))
        output = hltag(output)
        print(d1, "after highlighting:", len(output))

        with open(file_path + '_' + d1, "w") as f:
            json.dump(output, f, indent=2)

In [ ]:
preprocess(data_list1)

preprocessing: RACE-H/RACE-H_v1_trn.jsonl
sft_trn before highlighting: 62445
[['One hundred and thirteen million Americans have at least one bank-issued credit card. They give their owners automatic credit in stores, restaurants, and hotels, at home, across the country, and even abroad, and they make many banking services available as well. More and more of these credit cards can be read automatically, making it possible to withdraw or deposit money in scattered locations, whether or not the local branch bank is open. For many of us the "cashless society" is not on the horizon----it\'s already here.\nWhile computers offer these conveniences to consumers, they have many advantages for sellers too. Electronic cash registers can do much more than simply _ . They can keep a wide range of records, including who sold what, when, and to whom. This information allows businessmen to keep track of their list of goods by showing which items are being sold and how fast they are moving. Decisions t

In [ ]:
preprocess(data_list2)

preprocessing: SAT_ACT/SATACT_v3_trn.jsonl
sft_trn before highlighting: 919
[['Many of William Shakespeare’s tragedies address broad themes that still appeal to today’s audiences. For instance, Romeo and Juliet, which is set in the Italy of Shakespeare’s time, tackles the themes of parents versus children and love versus hate, and the play continues to be read and produced widely around the world. But understanding Shakespeare’s so-called history plays can require a knowledge of several centuries of English history. Consequently,   _  '], [{'question': 'many theatergoers and readers today are likely to find Shakespeare’s history plays less engaging than the tragedies.', 'choice': ['some of Shakespeare’s tragedies are more relevant to today’s audiences than twentieth-century plays.', 'Romeo and Juliet is the most thematically accessible of all Shakespeare’s tragedies.', 'experts in English history tend to prefer Shakespeare’s history plays to his other works.', None], 'answer': 'some of

In [6]:
def read_contexts(split):
  contexts = []
  with open(data_dir + split, "r") as f:
    for line in f:
      obj = json.loads(line)
      contexts.append(obj["context"])
  return contexts

In [7]:
def problem_gen(article, id):
    delimiter = '_[[#@]]_'

    def get_cloze(sentence, words):
        cloze = []
        ans = []
        dis = []
        sentences = sentence.split(delimiter)
        tokens = []

        for i, sent in enumerate(sentences):
            tokens += nltk.word_tokenize(sent)
            if i != len(sentences) - 1:
                tokens.append(delimiter)

        if len(tokens) > 50:
            return None

        used = set()
        n_cloze = min((len(tokens)-2) // 6, 4)

        if len(tokens) >= 6 and n_cloze <= 0:
            if len(tokens) >= 6:
                n_cloze = 1

        if n_cloze <= 0:
            return None

        n_cloze = random.randint(1, n_cloze)

        for _ in range(n_cloze):
            while True:
                cloze_len = random.randint(1, 4)
                left = random.randint(0, len(tokens)-cloze_len)

                if any(j in used for j in range(left, left + cloze_len)):
                  continue

                if not all(tokens[j].isalpha() for j in range(left, left + cloze_len)):
                  continue

                for j in range(left, left + cloze_len):
                  used.add(j)

                cloze.append([left, left + cloze_len])
                ans.append(' '.join(tokens[left:left + cloze_len]))
                break

        if not ans:
            return None

        for a in ans:
            dislen = max(1, random.randint(len(a.split()) - 1, len(a.split()) + 1))
            dislis = []
            for _ in range(3):
                while True:
                    start = random.randint(0, len(words)-dislen)
                    d = ' '.join(words[start:start+dislen])
                    if d != a and d not in dislis:
                        dislis.append(d)
                        break

            dis.append(dislis)

        for left, right in cloze:
            for j in range(left, right):
                tokens[j] = ''
            tokens[left] = '_'

        ret = [' '.join(' '.join(tokens).split()), ', '.join(ans)]
        for i in range(3):
            ret.append(', '.join(dis[i] for dis in dis))

        return ret

    d = [[], [], id]
    article = article.replace(delimiter, '')
    sentences_raw = sent_tokenize(article)
    sentences = [[s, idx] for idx, s in enumerate(sentences_raw)]
    words = [x for x in nltk.word_tokenize(article) if x.isalpha()]

    n_problem = min(10, len(words) // 30)
    for _ in tqdm(range(n_problem), desc=f"Generating problems for {id}"):
        random.shuffle(sentences)
        selected = [s[0] for s in sentences[:random.randint(1, 3)]]
        question = get_cloze(delimiter.join(selected), words)

        if question is not None:
            q = {"question": ' '.join(question[0].replace(delimiter, '').split()), "choice": question[1:]}
            if any(existing["question"] == q["question"] for existing in d[1]):
                continue

            if len(set(q["choice"])) != 4:
                continue

            random.shuffle(q["choice"])
            q["answer"] = question[1]
            d[1].append(q)

    sentences.sort(key = lambda x : x[1])
    d[0].append(' '.join([s[0] for s in sentences]))
    return d

In [32]:
cloze_data1 = ["RACE-H/RACE-H_v1_trn.jsonl"]
cloze_split = ["sftc_trn"]
#cloze_data1 = ["RACE-H/RACE-H_v1_dev.jsonl"]
#cloze_split = ["sftc_dev"]

for n in range(len(cloze_data1)):
  fn = cloze_split[n]
  dn = cloze_data1[n]
  output = []
  data = read_contexts(dn)
  #data = data[5000:6982] #skip idx 6982
  #data = data[6983:10000]
  #data = data[15000:15358] #skip idx 15358
  #data = data[15359:20000]
  #data = data[25000:26621] #skip idx 26621
  #data = data[26622:28297] #skip idx 28297
  #data = data[28298:29817]
  #data = data[29818:30000]
  #data = data[45000:49859] #skip idx 49859
  #data = data[49860:50000]
  #data = data[60000:65000] #END

"""
  for i, context in enumerate(data):
    output.append(problem_gen(context, str(i)))

  print(fn, "generated:", len(output), "total questions:", sum(len(item[1]) for item in output))
  output = hltag(output)

  print(fn, "after highlighting:", len(output))

  with open(data_dir + dn + '_' + fn + '_60000-64999', "w") as f:
    json.dump(output, f, indent=2)
"""

You may have heard the term "the American Dream". In 1848, James W. Marshall found gold in California and people began having golden dreams. That 19th century "American Dream" motivated   the Gold Rush and gave California its nickname of the "Golden State".
The American Dream drove not only 1800s gold-rush prospectors but also waves of immigrants throughout that century and the next. People from Europe, and a large number of Chinese, arrived in the US in the 19th century hoping that in America they would find gold in the streets. But most, instead, worked as railroad labourers. They created the oldest Chinatown, in San Francisco, and gave the city a Chinese name "the old gold hill".
In the 20th century, some critics said that it was no longer possible to become prosperous through determination and hard work. Unfair education for students from poor families and racial discrimination almost made the American Dream a nightmare.
Then, in the 1990s, California saw a new wave of dreamers in 

'\n  for i, context in enumerate(data):\n    output.append(problem_gen(context, str(i)))\n\n  print(fn, "generated:", len(output), "total questions:", sum(len(item[1]) for item in output))\n  output = hltag(output)\n\n  print(fn, "after highlighting:", len(output))\n\n  with open(data_dir + dn + \'_\' + fn + \'_60000-64999\', "w") as f:\n    json.dump(output, f, indent=2)\n'

In [ ]:
copyfile(data_dir + "RACE-H/sftc_RACE-H_v1_dev.jsonl", data_dir + "RACE-H/sftc_RACE-H_v1_tst.jsonl")

'/content/drive/MyDrive/NLU/Final Project/NLU_FinalProject/Data/JSONL_Formatted/RACE-H/sftc_RACE-H_v1_tst.jsonl'

In [ ]:
cloze_data2 = ["SAT_ACT/SATACT_v3_trn.jsonl", "SAT_ACT/SATACT_v3_dev.jsonl"]
cloze_split = ["sftc_trn", "sftc_dev"]

for n in range(len(cloze_data2)):
  fn = cloze_split[n]
  dn = cloze_data2[n]
  output = []
  data = read_contexts(dn)

  for i, context in enumerate(data):
    output.append(problem_gen(context, str(i)))

  print(fn, "generated:", len(output), "total questions:", sum(len(item[1]) for item in output))

  output = hltag(output)

  print(fn, "after highlighting:", len(output))

  with open(data_dir + dn + '_' + fn, "w") as f:
    json.dump(output, f, indent=2)

sftc_trn generated: 919 total questions: 1075
sftc_trn after highlighting: 1075
sftc_dev generated: 131 total questions: 161
sftc_dev after highlighting: 161


In [ ]:
copyfile(data_dir + "SAT_ACT/sftc2_SATACT_v3_dev.jsonl", data_dir + "SAT_ACT/sftc2_SATACT_v3_tst.jsonl")

'/content/drive/MyDrive/NLU/Final Project/NLU_FinalProject/Data/JSONL_Formatted/SAT_ACT/sftc2_SATACT_v3_tst.jsonl'

In [ ]:
import tensorflow as tf
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [ ]:
import re
import ftfy
import json
import spacy

from tqdm import tqdm

def get_pairs(word):
    """
    Return set of symbol pairs in a word.
    word is represented as tuple of symbols (symbols being variable-length strings)
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

def text_standardize(text):
    """
    fixes some issues the spacy tokenizer had on books corpus
    also does some whitespace standardization
    """
    text = text.replace('—', '-')
    text = text.replace('–', '-')
    text = text.replace('―', '-')
    text = text.replace('…', '...')
    text = text.replace('´', "'")
    text = re.sub('''(-+|~+|!+|"+|;+|\?+|\++|,+|\)+|\(+|\\+|\/+|\*+|\[+|\]+|}+|{+|\|+|_+)''', r' \1 ', text)
    text = re.sub('\s*\n\s*', ' \n ', text)
    text = re.sub('[^\S\n]+', ' ', text)
    return text.strip()

class TextEncoder(object):
    """
    mostly a wrapper for a public python bpe tokenizer
    """

    def __init__(self, model_name='gpt2-xl'):
        self.nlp = spacy.load('en_core_web_sm', disable=['parser', 'tagger', 'ner', 'textcat'])
        self.encoder = json.load(open(encoder_path))
        self.decoder = {v:k for k,v in self.encoder.items()}
        merges = open(bpe_path, encoding='utf8').read().split('\n')[1:-1]
        merges = [tuple(merge.split()) for merge in merges]
        self.bpe_ranks = dict(zip(merges, range(len(merges))))
        self.cache = {}

    def bpe(self, token):
        word = tuple(token[:-1]) + ( token[-1] + '</w>',)
        if token in self.cache:
            return self.cache[token]
        pairs = get_pairs(word)

        if not pairs:
            return token+'</w>'

        while True:
            bigram = min(pairs, key = lambda pair: self.bpe_ranks.get(pair, float('inf')))
            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j
                except:
                    new_word.extend(word[i:])
                    break

                if word[i] == first and i < len(word)-1 and word[i+1] == second:
                    new_word.append(first+second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word = tuple(new_word)
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)
        word = ' '.join(word)
        if word == '\n  </w>':
            word = '\n</w>'
        self.cache[token] = word
        return word

    def encode(self, texts, verbose=True):
        texts_tokens = []
        if verbose:
            for textraw in tqdm(texts, ncols=80, leave=False):
                text_tokens = []
                for text in textraw.split('[[HL]]'):
                    if '[[/HL]]' in text:
                        text_tokens.append('[[HL]]')
                        text,text2 = text.split('[[/HL]]')
                        text = self.nlp(text_standardize(ftfy.fix_text(text)))
                        for token in text:
                            text_tokens.extend(
                                [self.encoder.get(t, 0) for t in self.bpe(token.text.lower()).split(' ')])
                        text_tokens.append('[[/HL]]')
                        text = text2
                    text = self.nlp(text_standardize(ftfy.fix_text(text)))
                    for token in text:
                        text_tokens.extend(
                            [self.encoder.get(t, 0) for t in self.bpe(token.text.lower()).split(' ')])
                texts_tokens.append(text_tokens)
        else:
            for textraw in texts:
                text_tokens = []
                for text in textraw.split('[[HL]]'):
                    if '[[/HL]]' in text:
                        text_tokens.append('[[HL]]')
                        text, text2 = text.split('[[/HL]]')
                        text = self.nlp(text_standardize(ftfy.fix_text(text)))
                        for token in text:
                            text_tokens.extend(
                                [self.encoder.get(t, 0) for t in self.bpe(token.text.lower()).split(' ')])
                        text_tokens.append('[[/HL]]')
                        text = text2
                    text = self.nlp(text_standardize(ftfy.fix_text(text)))
                    for token in text:
                        text_tokens.extend(
                            [self.encoder.get(t, 0) for t in self.bpe(token.text.lower()).split(' ')])
                texts_tokens.append(text_tokens)
        return texts_tokens

In [ ]:
text_encoder = TextEncoder('model/encoder_bpe_40000.json', 'model/vocab_40000.bpe')
encoder = text_encoder.encoder
n_vocab = len(text_encoder.encoder)

FileNotFoundError: [Errno 2] No such file or directory: 'model/encoder_bpe_40000.json'

In [ ]:
def get_hl(X1, hl1t, hl2t):
  H1 = []
  X1n = []
  for i in range(len(X1)):
      H = []
      X = []
      hl = 0
      for j in range(len(X1[i])):
          if X1[i][j] == '[[HL]]':
              hl = 1
          elif X1[i][j] == '[[/HL]]':
              hl = 0
          else:
              X += [X1[i][j]]
              if hl == 1:
                  H += [hl1t]
              else:
                  H += [hl2t]
      H1 += [H]
      X1n += [X]
  return X1n, H1

In [ ]:
def encode_dataset(*splits, encoder):
  encoded_splits = []
  for split in splits[0]:
      fields = []
      for field in split:
          if isinstance(field[0], str):
              field = encoder.encode(field)
          fields.append(field)
      encoded_splits.append(fields)
  return encoded_splits

In [ ]:
import os
import csv
import numpy as np
import json

from tqdm import tqdm

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

seed = 3535999445

def race(path):
  X, Y = [], []
  for i in range(3): #set
      X += [[]]
      Y += [[]]
      for j in range(9):
          X[i] += [[]]

  for k in range(3):
      if k == 2:
          y = 0
      with open(os.path.join(path, ["sftc_SATACT_v3_trn.json", "sftc_SATACT_v3_dev.json", "sftc_SATACT_v3_tst.json"][k]), "r") as f: #CHANGE
          data = json.load(f)
          for i in range(len(data)):
              s = data[i][0]
              for j in range(len(data[i][1])):
                  qid = ''.join(data[i][-1].split("/")[1:]) + "-" + str(j)
                  q = data[i][1][j]["question"]
                  X[k][0] += [s[0]]
                  X[k][6] += [s[1]]
                  X[k][7] += [s[2]]
                  X[k][8] += [s[3]]
                  X[k][1] += [q]
                  for l in range(4):
                      c = data[i][1][j]["choice"][l]
                      X[k][l+2] += [c]
                      if k != 2 and c == data[i][1][j]["answer"]:
                          y = l
                  Y[k] += [y]

  trX1, trX2, trX3, trX4, trX5, trX6, trX7, trX8, trX9 = X[0][0], X[0][1], X[0][2], X[0][3], X[0][4], X[0][5], X[0][6], X[0][7], X[0][8]
  vaX1, vaX2, vaX3, vaX4, vaX5, vaX6, vaX7, vaX8, vaX9 = X[1][0], X[1][1], X[1][2], X[1][3], X[1][4], X[1][5], X[1][6], X[1][7], X[1][8]
  teX1, teX2, teX3, teX4, teX5, teX6, teX7, teX8, teX9 = X[2][0], X[2][1], X[2][2], X[2][3], X[2][4], X[2][5], X[2][6], X[2][7], X[2][8]
  trY = np.asarray(Y[0], dtype=np.int32)
  vaY = np.asarray(Y[1], dtype=np.int32)
  teY = np.asarray(Y[2], dtype=np.int32)
  return (trX1, trX2, trX3, trX4, trX5, trX6, trX7, trX8, trX9, trY), (vaX1, vaX2, vaX3, vaX4, vaX5, vaX6, vaX7, vaX8, vaX9, vaY), (teX1, teX2, teX3, teX4, teX5, teX6, teX7, teX8, teX9)


In [ ]:
def transform_race(X1, X2, X3, X4, X5, X6, X7, X8, X9, H1, H7, H8, H9, reverse = False):
  n_batch = len(X1)
  xmb = np.zeros((n_batch, 4, n_ctx, 3), dtype=np.int32)
  mmb = np.zeros((n_batch, 4, n_ctx), dtype=np.float32)
  start = encoder['_start_']
  delimiter = encoder['_delimiter_']
  for i, (x1, x2, x3, x4, x5, x6, x7, x8, x9, h1, h7, h8, h9), in enumerate(zip(X1, X2, X3, X4, X5, X6, X7, X8, X9, H1, H7, H8, H9)):
      if reverse:
          x12 = [start]+x3[:max_len]+[delimiter]+x2[:max_len]+x1[:max_len+max_len2]+[clf_token]
          x13 = [start]+x4[:max_len]+[delimiter]+x2[:max_len]+x7[:max_len+max_len2]+[clf_token]
          x14 = [start]+x5[:max_len]+[delimiter]+x2[:max_len]+x8[:max_len+max_len2]+[clf_token]
          x15 = [start]+x6[:max_len]+[delimiter]+x2[:max_len]+x9[:max_len+max_len2]+[clf_token]
      else:
          x12 = [start]+x1[:max_len+max_len2]+x2[:max_len]+[delimiter]+x3[:max_len]+[clf_token]
          x13 = [start]+x7[:max_len+max_len2]+x2[:max_len]+[delimiter]+x4[:max_len]+[clf_token]
          x14 = [start]+x8[:max_len+max_len2]+x2[:max_len]+[delimiter]+x5[:max_len]+[clf_token]
          x15 = [start]+x9[:max_len+max_len2]+x2[:max_len]+[delimiter]+x6[:max_len]+[clf_token]
      x12, x13, x14, x15 = x12[:n_ctx], x13[:n_ctx], x14[:n_ctx], x15[:n_ctx]
      l12 = len(x12)
      l13 = len(x13)
      l14 = len(x14)
      l15 = len(x15)
      h1 = h1[:max_len+max_len2]
      h7 = h7[:max_len+max_len2]
      h8 = h8[:max_len+max_len2]
      h9 = h9[:max_len+max_len2]
      lh1 = len(h1)
      lh7 = len(h7)
      lh8 = len(h8)
      lh9 = len(h9)
      xmb[i, 0, :l12, 0] = x12
      xmb[i, 1, :l13, 0] = x13
      xmb[i, 2, :l14, 0] = x14
      xmb[i, 3, :l15, 0] = x15

      if reverse:
          lc = len(x1[:max_len+max_len2]) + 1
          lq = len(x2[:max_len]) + 1
          lo = [len(x3[:max_len]) + 1, len(x4[:max_len]) + 1, len(x5[:max_len]) + 1, len(x6[:max_len]) + 1]
          xmb[i, 0, lo[0]+lq:lo[0]+lq+lh1, 2] = h1
          xmb[i, 1, lo[1]+lq:lo[1]+lq+lh7, 2] = h7
          xmb[i, 2, lo[2]+lq:lo[2]+lq+lh8, 2] = h8
          xmb[i, 3, lo[3]+lq:lo[3]+lq+lh9, 2] = h9
      else:
          xmb[i, 0, 1:lh1+1, 2] = h1
          xmb[i, 1, 1:lh7+1, 2] = h7
          xmb[i, 2, 1:lh8+1, 2] = h8
          xmb[i, 3, 1:lh9+1, 2] = h9

      mmb[i, 0, :l12] = 1
      mmb[i, 1, :l13] = 1
      mmb[i, 2, :l14] = 1
      mmb[i, 3, :l15] = 1
  xmb[:, :, :, 1] = np.arange(n_vocab+n_special, n_vocab+n_special+n_ctx)
  return xmb, mmb

In [ ]:
data_path = data_dir + 'SAT_ACT/'

In [ ]:
(trX1, trX2, trX3, trX4, trX5, trX6, trX7, trX8, trX9, trY), (vaX1, vaX2, vaX3, vaX4, vaX5, vaX6, vaX7, vaX8, vaX9, vaY), (teX1, teX2, teX3, teX4, teX5, teX6, teX7, teX8, teX9) = encode_dataset(race(data_path), encoder=text_encoder)
encoder['_start_'] = len(encoder)
encoder['_delimiter_'] = len(encoder)
encoder['_classify_'] = len(encoder)
encoder['_hl1_'] = len(encoder)
encoder['_hl2_'] = len(encoder)
clf_token = encoder['_classify_']
trX1, trH1 = get_hl(trX1, encoder['_hl1_'], encoder['_hl2_'])
vaX1, vaH1 = get_hl(vaX1, encoder['_hl1_'], encoder['_hl2_'])
teX1, teH1 = get_hl(teX1, encoder['_hl1_'], encoder['_hl2_'])

trX7, trH7 = get_hl(trX7, encoder['_hl1_'], encoder['_hl2_'])
vaX7, vaH7 = get_hl(vaX7, encoder['_hl1_'], encoder['_hl2_'])
teX7, teH7 = get_hl(teX7, encoder['_hl1_'], encoder['_hl2_'])

trX8, trH8 = get_hl(trX8, encoder['_hl1_'], encoder['_hl2_'])
vaX8, vaH8 = get_hl(vaX8, encoder['_hl1_'], encoder['_hl2_'])
teX8, teH8 = get_hl(teX8, encoder['_hl1_'], encoder['_hl2_'])

trX9, trH9 = get_hl(trX9, encoder['_hl1_'], encoder['_hl2_'])
vaX9, vaH9 = get_hl(vaX9, encoder['_hl1_'], encoder['_hl2_'])
teX9, teH9 = get_hl(teX9, encoder['_hl1_'], encoder['_hl2_'])

n_special = 5
n_ctx = 512
max_len = n_ctx//2-2
n_ctx = max([len(x1[:max_len])+len(x2[:max_len])+max(len(x3[:max_len]), len(x4[:max_len]), len(x5[:max_len]), len(x6[:max_len])) for x1, x2, x3, x4, x5, x6 in zip(trX1, trX2, trX3, trX4, trX5, trX6)]+\
                  [len(x1[:max_len])+len(x2[:max_len])+max(len(x3[:max_len]), len(x4[:max_len]), len(x5[:max_len]), len(x6[:max_len])) for x1, x2, x3, x4, x5, x6 in zip(vaX1, vaX2, vaX3, vaX4, vaX5, vaX6)]+\
                  [len(x1[:max_len])+len(x2[:max_len])+max(len(x3[:max_len]), len(x4[:max_len]), len(x5[:max_len]), len(x6[:max_len])) for x1, x2, x3, x4, x5, x6 in zip(teX1, teX2, teX3, teX4, teX5, teX6)])+3
max_len2 = 512 - n_ctx
n_ctx = 512

reverse = False
submit = True
trX, trM = transform_race(trX1, trX2, trX3, trX4, trX5, trX6, trX7, trX8, trX9, trH1, trH7, trH8, trH9, reverse)
vaX, vaM = transform_race(vaX1, vaX2, vaX3, vaX4, vaX5, vaX6, vaX7, vaX8, vaX9, vaH1, vaH7, vaH8, vaH9, reverse)
if submit:
    teX, teM = transform_race(teX1, teX2, teX3, teX4, teX5, teX6, teX7, teX8, teX9, teH1, teH7, teH8, teH9, reverse)


# Archived Code

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if 'Salm00n' in model_name: # LoRA fine-tuned model
  config = PeftConfig.from_pretrained(model_name)
  base_model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path,
                                                    torch_dtype=torch.float16)
  model = PeftModel.from_pretrained(base_model, model_name, torch_dtype=torch.float16)
  model = model.merge_and_unload()
else: # base model
  model = AutoModelForCausalLM.from_pretrained(model_name)

model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id,
                              reduction="none")

adapter_config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/config.py:162: UserWarning: Unexpected keyword arguments ['alpha_pattern', 'bias', 'corda_config', 'eva_config', 'exclude_modules', 'fan_in_fan_out', 'init_lora_weights', 'layer_replication', 'layers_pattern', 'layers_to_transform', 'loftq_config', 'lora_alpha', 'lora_bias', 'lora_dropout', 'megatron_config', 'megatron_core', 'modules_to_save', 'r', 'rank_pattern', 'target_modules', 'trainable_token_indices', 'use_dora', 'use_rslora'] for class PeftConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

Load and Process Data

In [ ]:
with open(data_dir + data_path, 'r') as f:
  data = [json.loads(line) for line in f]

print(f'Number of Questions = {len(data)}\n')
print(data[0].keys())

Number of Questions = 3498

dict_keys(['context', 'question', 'answerA', 'answerB', 'answerC', 'answerD', 'correct'])


In [ ]:
def create_input(batch, sys_prompt='', fs_demos=''):
  texts = []
  for text in batch:
    pqa = [f"{fs_demos}Q: {text['context']} {text['question']}\nA:{sys_prompt} {text[i]}"
            for i in ['answerA', 'answerB', 'answerC', 'answerD']]
    texts.extend(pqa)

  return texts

create_input(data[:BATCH_SIZE])

['Q: According to the passage, we know that   _  .\nA: people with good facial features must be trustworthy',
 'Q: According to the passage, we know that   _  .\nA: people with bad facial features could not be trustworthy',
 'Q: According to the passage, we know that   _  .\nA: we should judge people by their facial features',
 'Q: According to the passage, we know that   _  .\nA: facial features might give people some wrong impressions']

Run Inference

In [ ]:
def batch_predict(batch_processed):
  input = tokenizer(batch_processed, padding=True, return_tensors='pt').to(device)

  # truncating if needed (truncate front of input)
  if input['input_ids'].shape[-1] > 1024:
    trunc = [True]* (len(batch_processed)//4)
    input['input_ids'] = input['input_ids'][:,-1024:]
    input['attention_mask'] = input['attention_mask'][:,-1024:]
  else:
    trunc = [False]* (len(batch_processed)//4)

  with torch.no_grad():
    output = model(**input, max_new_tokens=0)

  # get prediction with CrossEntropyLoss
  logits = output.logits[:, :-1, :].to('cpu')
  targets = input['input_ids'][:, 1:].to('cpu')

  loss = loss_fn(logits.permute(0, 2, 1), targets)
  total_loss = torch.sum(loss, dim=-1)
  total_loss = total_loss.view(-1, 4)

  preds = torch.argmin(total_loss, dim=-1).tolist()

  return trunc, total_loss.numpy(), preds

In [ ]:
# test output
example_batch = data[:BATCH_SIZE]
batch_predict(create_input(example_batch))

([False], array([[118.44, 122.25, 111.06, 126.56]], dtype=float16), [2])

In [ ]:
res = defaultdict(list)

for i in tqdm(range(0, len(data), BATCH_SIZE)):
  batch = data[i:i+BATCH_SIZE]
  try:
    trunc, total_loss, preds = batch_predict(create_input(batch))
  except Exception as e:
    print(e)
    trunc = [np.nan]*BATCH_SIZE
    preds = [np.nan]*BATCH_SIZE
    total_loss = np.full((BATCH_SIZE, 4), np.nan)

  res['pred'].extend(preds)
  res['truncated'].extend(trunc)
  res['loss_A'].extend(total_loss[:,0].tolist())
  res['loss_B'].extend(total_loss[:,1].tolist())
  res['loss_C'].extend(total_loss[:,2].tolist())
  res['loss_D'].extend(total_loss[:,3].tolist())

100%|██████████| 3498/3498 [03:37<00:00, 16.08it/s]


In [ ]:
res_df = pd.concat([pd.DataFrame(res), pd.DataFrame(data)], axis=1)
res_df['pred'] = res_df['pred'].apply(lambda x: 'ABCD'[x])
res_df.head()

,pred,truncated,loss_A,loss_B,loss_C,loss_D,context,question,answerA,answerB,answerC,answerD,correct
0,C,False,118.4375,122.2500,111.0625,126.5625,It is the goal of politicians everywhere-----h...,"According to the passage, we know that _ .",people with good facial features must be trust...,people with bad facial features could not be t...,we should judge people by their facial features,facial features might give people some wrong i...,D
1,D,False,137.1250,138.1250,133.3750,130.1250,It is the goal of politicians everywhere-----h...,"According to Ms Cornwell, we can infer that ...",the science will give politicians great help,politicians could be successful with the help ...,politicians won't think highly of the science,politicians will be satisfied with the science,C
2,B,False,82.6250,73.2500,76.1875,77.6875,It is the goal of politicians everywhere-----h...,What's the best title for the passage?,How Science could Help Politicians,How to Win the Trust of Voters,The Other Sides of Politicians,An Important Discovery for Politicians,A
3,B,False,104.8750,103.8125,104.2500,109.0625,"In the 1960s, people asked about your astrolog...",The main purpose of the passage is to tell you...,what a website is like,how to build your own website,how to meet people online,what a website is made up of,B
4,B,False,110.8125,106.9375,107.8750,111.5625,"In the 1960s, people asked about your astrolog...","According to the writer, your website is a pla...",where you can meet people all around the world,where you can buy what you want,where you can get free services,where you can meet people on the Internet,D


In [ ]:
print(res_df.pred.value_counts())

pred
A    1216
B     852
D     716
C     714
Name: count, dtype: int64


In [ ]:
# evaluation metrics

# accuracy
accuracy = sum(res_df.pred == res_df.correct)/len(res_df.dropna())

# percent failure
res_fail = sum(res_df.pred.isnull())/len(res_df)

# percent truncated
res_trunc = sum(res_df.truncated)/len(res_df)

df_eval = pd.DataFrame({'model':[model_name],
                        'dataset':[data_name],
                        'accuracy':[accuracy],
                        'response_failure':[res_fail],
                        'prompt_truncation':[res_trunc]})

if os.path.exists(f"{save_dir}benchmark_summary_v2.csv"):
  df_temp = pd.read_csv(f"{save_dir}benchmark_summary_v2.csv")
  df_eval = pd.concat([df_eval, df_temp], axis=0, ignore_index=True)

df_eval

,model,dataset,accuracy,response_failure,prompt_truncation
0,Salm00n/gpt2-xl_RACE-H_v1,RACE-H_InfoRemove_test,0.234706,0.0,0.000000
1,Salm00n/gpt2-xl_SATACT_v1,SATACT_InfoRemove_test,0.246212,0.0,0.000000
2,Salm00n/gpt2-xl_SATACT_v1,SATACT_test,0.303030,0.0,0.018939
3,Salm00n/gpt2-xl_RACE-H_v1,RACE-H_test,0.266724,0.0,0.003716
4,gpt2-xl,RACE-H_test,0.312464,0.0,0.003716
5,gpt2-xl,RACE-H_InfoRemove_test,0.240995,0.0,0.000000
6,gpt2-xl,SATACT_InfoRemove_test,0.246212,0.0,0.000000
7,gpt2-xl,SATACT_test,0.306818,0.0,0.018939


Store Results

In [ ]:
if '/' in model_name:
  model_name = model_name.split('/')[1]
  print(model_name)
res_df.to_csv(f"{save_dir}{data_name}_benchmark_{model_name}_v2.csv", index=False)

gpt2-xl_RACE-H_v1


In [ ]:
df_eval.to_csv(f"{save_dir}benchmark_summary_v2.csv", index=False)